In [1]:
# ── Installations (Google Colab) ────────────────────────────────────────────
!pip install bokeh --quiet


In [2]:
# ── Upload du dataset Melbourne ─────────────────────────────────────────────
from google.colab import files
uploaded = files.upload()   # Sélectionner : daily-minimum-temperatures-in-melbourne.csv


Saving daily-minimum-temperatures-in-melbourne.csv to daily-minimum-temperatures-in-melbourne.csv


In [3]:
# ── Imports & chargement des données ────────────────────────────────────────
import pandas as pd
import calendar
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import (
    ColumnDataSource, HoverTool, DatetimeTickFormatter,
    DateRangeSlider, CustomJS
)
from bokeh.layouts import column
from bokeh.transform import factor_cmap
from bokeh.palettes import Viridis10
from bokeh.io import output_notebook

output_notebook()

# Chargement
df = pd.read_csv("daily-minimum-temperatures-in-melbourne.csv")
df.columns = ['Date', 'Temperature']
df['Date'] = pd.to_datetime(df['Date'])
df['Temperature'] = df['Temperature'].astype(str).str.replace('?', '', regex=False)
df['Temperature'] = pd.to_numeric(df['Temperature'], errors='coerce')
df.dropna(subset=['Temperature'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Dataset chargé : {len(df)} lignes, de {df['Date'].min().date()} à {df['Date'].max().date()}")
df.head()


Dataset chargé : 3650 lignes, de 1981-01-01 à 1990-12-31


,Date,Temperature
0,1981-01-01,20.7
1,1981-01-02,17.9
2,1981-01-03,18.8
3,1981-01-04,14.6
4,1981-01-05,15.8


## Question 1 : Graphique linéaire de série temporelle de base

In [4]:
# Answer 1 : Basic Time Series Line Plot

source1 = ColumnDataSource(df)

hover1 = HoverTool(
    tooltips=[
        ("Date",        "@Date{%F}"),
        ("Température", "@Temperature{0.0} °C"),
    ],
    formatters={"@Date": "datetime"},
    mode="vline"
)

p1 = figure(
    title="Daily Minimum Temperatures",
    x_axis_label="Date",
    y_axis_label="Temperature (°C)",
    x_axis_type="datetime",
    tools=["pan", "wheel_zoom", "reset"],
    width=900, height=400,
)
p1.add_tools(hover1)
p1.line("Date", "Temperature", source=source1,
        color="steelblue", line_width=1.5, line_alpha=0.8)
p1.xaxis.formatter = DatetimeTickFormatter(months="%b %Y", years="%Y")
p1.title.text_font_size = "14pt"

show(p1)


## Question 2 : Moyenne mobile sur 30 jours

In [5]:
# Answer 2 : Rolling Average

df['Rolling_Avg'] = df['Temperature'].rolling(window=30, min_periods=1).mean()
source2 = ColumnDataSource(df)

hover2 = HoverTool(
    tooltips=[
        ("Date",          "@Date{%F}"),
        ("Température",   "@Temperature{0.0} °C"),
        ("Moy. 30 jours", "@Rolling_Avg{0.0} °C"),
    ],
    formatters={"@Date": "datetime"},
    mode="vline"
)

p2 = figure(
    title="Daily Minimum Temperatures with 30-Day Rolling Average",
    x_axis_label="Date",
    y_axis_label="Temperature (°C)",
    x_axis_type="datetime",
    tools=["pan", "wheel_zoom", "reset"],
    width=900, height=420,
)
p2.add_tools(hover2)

p2.line("Date", "Temperature", source=source2,
        color="steelblue", line_width=1.2, alpha=0.5,
        legend_label="Température quotidienne")

p2.line("Date", "Rolling_Avg", source=source2,
        color="firebrick", line_width=2.5,
        legend_label="Moyenne mobile 30 j")

p2.legend.location     = "top_left"
p2.legend.click_policy = "hide"
p2.xaxis.formatter = DatetimeTickFormatter(months="%b %Y", years="%Y")

show(p2)


## Question 3 : Boîtes à moustaches mensuelles

In [6]:
# Answer 3 : Monthly Box Plots

df['Month'] = df['Date'].dt.month

grp3 = df.groupby('Month')['Temperature']
stats3 = grp3.agg(
    min    ='min',
    max    ='max',
    median ='median',
    q1     = lambda x: x.quantile(0.25),
    q3     = lambda x: x.quantile(0.75),
).reset_index()
stats3['month_name'] = stats3['Month'].apply(lambda m: calendar.month_abbr[m])

source3 = ColumnDataSource(stats3)
months  = [calendar.month_abbr[i] for i in range(1, 13)]

hover3 = HoverTool(tooltips=[
    ("Mois",    "@month_name"),
    ("Min",     "@min{0.0} °C"),
    ("Q1",      "@q1{0.0} °C"),
    ("Médiane", "@median{0.0} °C"),
    ("Q3",      "@q3{0.0} °C"),
    ("Max",     "@max{0.0} °C"),
])

p3 = figure(
    title="Distribution mensuelle des températures – Melbourne",
    x_axis_label="Month", y_axis_label="Temperature (°C)",
    x_range=months,
    tools=["pan", "wheel_zoom", "reset"],
    width=900, height=450,
)
p3.add_tools(hover3)

p3.vbar("month_name", top="q3", bottom="q1", width=0.6,
        source=source3, fill_color="steelblue", line_color="navy", fill_alpha=0.7)

p3.rect(x="month_name", y="median", width=0.6, height=0.08,
        source=source3, fill_color="firebrick", line_color="firebrick")

p3.segment("month_name", "q3", "month_name", "max",
           source=source3, line_color="black", line_width=1.5)
p3.segment("month_name", "q1", "month_name", "min",
           source=source3, line_color="black", line_width=1.5)

p3.rect(x="month_name", y="max", width=0.3, height=0.05,
        source=source3, line_color="black", fill_color="black")
p3.rect(x="month_name", y="min", width=0.3, height=0.05,
        source=source3, line_color="black", fill_color="black")

show(p3)


## Question 4 : Boîtes à moustaches annuelles avec color mapping

In [7]:
# Answer 4 : Annual Box Plots with factor_cmap

df['Year'] = df['Date'].dt.year

grp4 = df.groupby('Year')['Temperature']
stats4 = grp4.agg(
    min    ='min',
    max    ='max',
    median ='median',
    q1     = lambda x: x.quantile(0.25),
    q3     = lambda x: x.quantile(0.75),
    mean   ='mean',
).reset_index()
stats4['Year_str'] = stats4['Year'].astype(str)

years   = stats4['Year_str'].tolist()
source4 = ColumnDataSource(stats4)

palette = Viridis10[:len(years)]
cmap    = factor_cmap('Year_str', palette=palette, factors=years)

hover4 = HoverTool(tooltips=[
    ("Année",   "@Year"),
    ("Min",     "@min{0.0} °C"),
    ("Q1",      "@q1{0.0} °C"),
    ("Médiane", "@median{0.0} °C"),
    ("Q3",      "@q3{0.0} °C"),
    ("Max",     "@max{0.0} °C"),
    ("Moyenne", "@mean{0.0} °C"),
])

p4 = figure(
    title="Distribution annuelle des températures – Melbourne",
    x_axis_label="Year", y_axis_label="Temperature (°C)",
    x_range=years,
    tools=["pan", "wheel_zoom", "reset"],
    width=900, height=450,
)
p4.add_tools(hover4)

p4.vbar("Year_str", top="q3", bottom="q1", width=0.6,
        source=source4, fill_color=cmap, line_color="black", fill_alpha=0.85)

p4.rect(x="Year_str", y="median", width=0.6, height=0.08,
        source=source4, fill_color="white", line_color="black", line_width=2)

p4.segment("Year_str", "q3", "Year_str", "max",
           source=source4, line_color="black", line_width=1.5)
p4.segment("Year_str", "q1", "Year_str", "min",
           source=source4, line_color="black", line_width=1.5)

p4.rect(x="Year_str", y="max", width=0.3, height=0.05,
        source=source4, line_color="black", fill_color="black")
p4.rect(x="Year_str", y="min", width=0.3, height=0.05,
        source=source4, line_color="black", fill_color="black")

show(p4)


## Question 5 : Sélection interactive de la plage temporelle

In [8]:
# Answer 5 : Interactive Date Range Selection with DateRangeSlider + CustomJS

source5_full = ColumnDataSource(df)
source5_view = ColumnDataSource(df.copy())

start_ts = int(df['Date'].min().timestamp() * 1000)
end_ts   = int(df['Date'].max().timestamp() * 1000)

hover5 = HoverTool(
    tooltips=[
        ("Date",        "@Date{%F}"),
        ("Température", "@Temperature{0.0} °C"),
    ],
    formatters={"@Date": "datetime"},
    mode="vline"
)

p5 = figure(
    title="Températures minimales – Sélection interactive de la plage",
    x_axis_label="Date", y_axis_label="Temperature (°C)",
    x_axis_type="datetime",
    tools=["pan", "wheel_zoom", "reset"],
    width=900, height=400,
)
p5.add_tools(hover5)
p5.line("Date", "Temperature", source=source5_view,
        color="steelblue", line_width=1.5, line_alpha=0.8)
p5.xaxis.formatter = DatetimeTickFormatter(months="%b %Y", years="%Y")

slider5 = DateRangeSlider(
    title="Sélectionner la période",
    value=(start_ts, end_ts),
    start=start_ts, end=end_ts,
    step=24*60*60*1000,
    width=880,
)

callback5 = CustomJS(
    args=dict(src_full=source5_full, src_view=source5_view),
    code="""
        const [start, end] = cb_obj.value;
        const dates = src_full.data['Date'];
        const temps = src_full.data['Temperature'];
        const new_dates = [], new_temps = [];
        for (let i = 0; i < dates.length; i++) {
            if (dates[i] >= start && dates[i] <= end) {
                new_dates.push(dates[i]);
                new_temps.push(temps[i]);
            }
        }
        src_view.data = {Date: new_dates, Temperature: new_temps};
        src_view.change.emit();
    """
)
slider5.js_on_change('value', callback5)

show(column(p5, slider5))


## Question 6 : Décomposition de série temporelle

In [9]:
# Answer 6 : Time Series Decomposition Visualization

monthly = (df.set_index('Date')['Temperature']
             .resample('ME').mean()
             .reset_index())
monthly.columns = ['Date', 'Monthly_Avg']
monthly['Trend']    = monthly['Monthly_Avg'].rolling(window=12, center=True).mean()
monthly['Seasonal'] = monthly['Monthly_Avg'] - monthly['Trend']

src6 = ColumnDataSource(monthly)
fmt  = DatetimeTickFormatter(months="%b %Y", years="%Y")

def make_hover(col):
    return HoverTool(
        tooltips=[("Date", "@Date{%b %Y}"), (col, f"@{{{col}}}{{0.00}}")],
        formatters={"@Date": "datetime"}, mode="vline"
    )

p6a = figure(title="Données mensuelles", x_axis_type="datetime",
             x_axis_label="Date", y_axis_label="Temp. moy. (°C)",
             tools="pan,wheel_zoom,reset", width=900, height=250)
p6a.add_tools(make_hover("Monthly_Avg"))
p6a.line("Date", "Monthly_Avg", source=src6, color="steelblue", line_width=2)
p6a.xaxis.formatter = fmt

p6b = figure(title="Tendance (moyenne mobile 12 mois)", x_axis_type="datetime",
             x_range=p6a.x_range,
             x_axis_label="Date", y_axis_label="Tendance (°C)",
             tools="pan,wheel_zoom,reset", width=900, height=250)
p6b.add_tools(make_hover("Trend"))
p6b.line("Date", "Trend", source=src6, color="firebrick", line_width=2.5)
p6b.xaxis.formatter = fmt

p6c = figure(title="Saisonnalité (données – tendance)", x_axis_type="datetime",
             x_range=p6a.x_range,
             x_axis_label="Date", y_axis_label="Saisonnalité (°C)",
             tools="pan,wheel_zoom,reset", width=900, height=250)
p6c.add_tools(make_hover("Seasonal"))
p6c.line("Date", "Seasonal", source=src6, color="seagreen", line_width=2)
p6c.segment(x0="Date", y0=0, x1="Date", y1="Seasonal",
            source=src6, line_color="seagreen", line_alpha=0.4)
p6c.xaxis.formatter = fmt

show(column(p6a, p6b, p6c))
